In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import numpy as np
import os

try:
    from py_vncorenlp import VnCoreNLP
    rdrsegmenter = VnCoreNLP("https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar", annotators="wseg,pos,ner,parse", max_heap_size='-Xmx2g')
    USE_VNCORENLP = True
except:
    USE_VNCORENLP = False
    import re

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
phobert = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(
            encoder.config.hidden_size, 1
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        scores = self.classifier(cls_embeddings)
        return scores
model = PhoBERTSUM(phobert).to(DEVICE)
model.eval()   # inference mode
def sent_tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []
    
    if USE_VNCORENLP:
        try:
            sentences = rdrsegmenter.sent_tokenize(text)
            sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
            return sentences
        except:
            pass
    
    sentences = re.split(r'[.!?]+\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

def split_sentences(text):
    if not isinstance(text, str):
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]
def get_top_k_by_ratio(num_sentences, ratio=0.6):
    return max(1, int(num_sentences * ratio))
def encode_sentences(sentences, max_len=256):
    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    return encoded["input_ids"].to(DEVICE), encoded["attention_mask"].to(DEVICE)
@torch.no_grad()
def extractive_summary(text, ratio=0.6):
    sentences = split_sentences(text)
    n = len(sentences)

    if n == 0:
        return ""

    # Nếu chỉ có 1 câu → lấy luôn
    if n == 1:
        return sentences[0]

    top_k = get_top_k_by_ratio(n, ratio)

    input_ids, attention_mask = encode_sentences(sentences)
    scores = model(input_ids, attention_mask)

    scores = scores.squeeze().cpu()

    # ĐẢM BẢO top_idx LUÔN là list
    top_idx = torch.topk(
        scores, min(top_k, n)
    ).indices.tolist()

    if not isinstance(top_idx, list):
        top_idx = [top_idx]

    top_idx.sort()

    summary = " ".join(sentences[i] for i in top_idx)
    return summary

rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def evaluate_summary(candidate, reference):
    if not candidate or not reference:
        return {
            'f1': 0.0,
            'rouge1': 0.0,
            'rougel': 0.0,
            'bertscore': 0.0
        }
    
    rouge_scores = rouge_scorer_obj.score(reference, candidate)
    rouge1 = rouge_scores['rouge1'].fmeasure
    rougel = rouge_scores['rougeL'].fmeasure
    
    try:
        P, R, F1_bert = bert_score([candidate], [reference], lang='vi', verbose=False)
        bertscore = F1_bert.item()
    except:
        bertscore = 0.0
    
    candidate_tokens = set(candidate.lower().split())
    reference_tokens = set(reference.lower().split())
    if len(reference_tokens) == 0:
        f1 = 0.0
    else:
        intersection = candidate_tokens & reference_tokens
        precision = len(intersection) / len(candidate_tokens) if len(candidate_tokens) > 0 else 0
        recall = len(intersection) / len(reference_tokens) if len(reference_tokens) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'f1': f1,
        'rouge1': rouge1,
        'rougel': rougel,
        'bertscore': bertscore
    }

def select_best_summary(summaries_dict):
    if not summaries_dict:
        return None, {}
    
    best_key = None
    best_scores = None
    
    for key, scores in summaries_dict.items():
        if best_key is None:
            best_key = key
            best_scores = scores
            continue
        
        if scores['f1'] > best_scores['f1']:
            best_key = key
            best_scores = scores
        elif scores['f1'] == best_scores['f1']:
            if scores['rougel'] > best_scores['rougel']:
                best_key = key
                best_scores = scores
            elif scores['rougel'] == best_scores['rougel']:
                if scores['rouge1'] > best_scores['rouge1']:
                    best_key = key
                    best_scores = scores
                elif scores['rouge1'] == best_scores['rouge1']:
                    if scores['bertscore'] > best_scores['bertscore']:
                        best_key = key
                        best_scores = scores
    
    return best_key, best_scores

In [ ]:
INPUT_XLSX = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\DATA_TX.xlsx"
OUTPUT_XLSX = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\DATA_TX_summary.xlsx"

df = pd.read_excel(INPUT_XLSX)

if 'f1' not in df.columns:
    df['f1'] = np.nan
if 'rougel' not in df.columns:
    df['rougel'] = np.nan
if 'rouge1' not in df.columns:
    df['rouge1'] = np.nan
if 'bertscore' not in df.columns:
    df['bertscore'] = np.nan

reference_col = None
for col in ['reference_summary', 'gold_summary', 'reference', 'gold']:
    if col in df.columns:
        reference_col = col
        break

for idx, row in df.iterrows():
    content = row["content"]
    if pd.isna(content) or content == "":
        continue
    
    reference = ""
    if reference_col and reference_col in row:
        ref_val = row[reference_col]
        if not pd.isna(ref_val) and ref_val != "":
            reference = str(ref_val)
    elif not pd.isna(row.get("summary", "")) and row.get("summary", "") != "":
        reference = str(row["summary"])
    
    best_summary = None
    best_scores = None
    
    for retry in range(5):
        summary = extractive_summary(content, ratio=0.6)
        
        if reference and reference.strip():
            scores = evaluate_summary(summary, reference)
        else:
            scores = {'f1': 0.0, 'rouge1': 0.0, 'rougel': 0.0, 'bertscore': 0.0}
        
        if (scores['f1'] >= 0.5 and 
            scores['rouge1'] >= 0.45 and 
            scores['rougel'] >= 0.4 and 
            scores['bertscore'] >= 0.8):
            best_summary = summary
            best_scores = scores
            break
    
    if best_summary is not None:
        df.at[idx, "summary"] = best_summary
        df.at[idx, "f1"] = best_scores['f1']
        df.at[idx, "rougel"] = best_scores['rougel']
        df.at[idx, "rouge1"] = best_scores['rouge1']
        df.at[idx, "bertscore"] = best_scores['bertscore']

df.to_excel(OUTPUT_XLSX, index=False)

print("✅ Đã tạo xong bản tóm tắt trích xuất")
